# 1회차: Vector·Linear combination ($\mathbb{R}^n$)

> 사전 reading: Strang §1.1 · 3Blue1Brown EoLA Ch.1
> **이 노트북의 한 줄**: 정의 1.4 Linear combination이 곧 **신경망 한 층 $W\mathbf{x}+\mathbf{b}$**임을 코드와 그림으로 본다.

**오늘 보일 것 (정의 1.1~1.5)**: ① 덧셈·Scalar곱의 기하 ② Linear combination·Span(독립이면 평면, 평행이면 직선) ③ **신경망 한 층 = $W$ 열들의 가중 Linear combination**

In [ ]:
# Colab 한글 폰트 (matplotlib 라벨 깨짐 방지)
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # 한글 폰트 자동 등록
import numpy as np
import matplotlib.pyplot as plt

# 이 노트북은 난수를 쓰지 않지만, 습관적으로 시드를 고정해 둔다(재현성).
np.random.seed(0)

## 1. Vector·덧셈·Scalar곱 (정의 1.1~1.3)

$\mathbf{x}=(x_1,\ldots,x_n)^\top\in\mathbb{R}^n$ (Vector는 굵게, Scalar는 보통체 $\alpha$).
- **덧셈** $\mathbf{x}+\mathbf{y}$: 성분별 합 (차원 다르면 미정의)
- **Scalar곱** $\alpha\mathbf{x}$: 성분별 Scalar 곱

> 두 연산의 자연스러운 성질(교환·결합·분배 등)은 **6회차에서 vector space 8 axioms로 정식화**한다. 오늘은 계산과 기하 직관만.

In [ ]:
# R^4의 두 벡터. 오늘의 주인공은 '덧셈과 Scalar곱을 섞은 식' = Linear combination이다.
a = np.array([3, 1, -2, 4], dtype=float)
b = np.array([-1, 2, 5, 0], dtype=float)

# 아래 세 줄은 모두 a, b의 Linear combination이다(계수만 다르다).
# NumPy는 +와 *를 성분별로 적용하므로, 정의 1.2(덧셈)·1.3(Scalar곱)이 코드 한 줄로 그대로 옮겨진다.
print('a + b   =', a + b)          # 계수 (1, 1)
print('2a      =', 2 * a)          # 계수 (2, 0)
print('2a - 3b =', 2 * a - 3 * b)  # 계수 (2, -3): 덧셈·Scalar곱을 한 번에 섞은 식

In [ ]:
# [그림 1] 두 연산의 기하: 덧셈 = 평행사변형, Scalar곱 = 같은 직선 위 신축·반전 (R^2에서 눈으로 확인)
u = np.array([2., 1.]); v = np.array([-1., 2.])

# 화살표 하나를 그리는 헬퍼. origin에서 시작해 vec만큼 가는 화살표와 라벨을 그린다(순수 시각화용).
def arrow(ax, vec, color, label, origin=(0, 0)):
    ax.annotate('', xy=(origin[0]+vec[0], origin[1]+vec[1]), xytext=origin,
                arrowprops=dict(arrowstyle='-|>', color=color, lw=2.2))
    ax.text(origin[0]+vec[0]*1.06, origin[1]+vec[1]*1.06, label,
            color=color, fontsize=13, fontweight='bold')

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
# (왼쪽) 덧셈: u와 v를 그린 뒤 합 u+v를 그린다.
arrow(ax[0], u, '#2563EB', r'$\mathbf{u}$')
arrow(ax[0], v, '#059669', r'$\mathbf{v}$')
arrow(ax[0], u+v, '#DC2626', r'$\mathbf{u}+\mathbf{v}$')
# v를 u의 끝으로, u를 v의 끝으로 평행이동해 회색 보조선을 그리면 평행사변형이 완성된다.
# 그 대각선이 곧 u+v → '덧셈 = 평행사변형 법칙'이 눈에 보인다.
arrow(ax[0], v, '#9CA3AF', '', origin=u)
arrow(ax[0], u, '#9CA3AF', '', origin=v)
ax[0].set_title('덧셈 = 평행사변형 법칙')
# (오른쪽) Scalar곱: 같은 u에 2를 곱하면 방향 그대로 2배로 늘고, -1을 곱하면 방향이 반대로 뒤집힌다.
# 세 화살표가 모두 한 직선 위에 놓이는 게 핵심 — Scalar곱은 방향을 바꾸지 않고 크기·부호만 바꾼다.
arrow(ax[1], u, '#2563EB', r'$\mathbf{u}$')
arrow(ax[1], 2*u, '#DC2626', r'$2\mathbf{u}$')
arrow(ax[1], -u, '#7C3AED', r'$-\mathbf{u}$')
ax[1].set_title('Scalar곱 = 같은 직선 위 신축·반전')
# 두 그림 모두 축 비율을 1:1로 맞춰야(set_aspect) 각도·길이가 왜곡 없이 보인다.
for a_ in ax:
    a_.axhline(0, color='#ddd', lw=.8); a_.axvline(0, color='#ddd', lw=.8)
    a_.set_xlim(-4, 5); a_.set_ylim(-3, 4); a_.set_aspect('equal'); a_.grid(alpha=.3)
    a_.set_xlabel('$x_1$'); a_.set_ylabel('$x_2$')
plt.tight_layout(); plt.show()

## 2. Linear combination·Span (정의 1.4~1.5)

- **Linear combination**: $\alpha_1\mathbf{v}_1+\cdots+\alpha_k\mathbf{v}_k$ ($\alpha_i$ = 계수)
- **Span(직관)**: 그 Vector들로 만들 수 있는 모든 Linear combination의 집합 (정식 정의 8회차)

> 핵심: **Vector 개수 ≠ Span 차원.** 두 Vector가 독립이면 span은 평면, 평행이면 직선이다.

In [ ]:
# [그림 2] Span을 '계수를 전부 대입해 본다'로 시각화한다.
# 계수 (α, β)를 -2~2 격자로 촘촘히 잡고, 각 계수쌍이 만드는 점 αv1+βv2를 전부 찍는다.
g = np.linspace(-2, 2, 21)
A, B = np.meshgrid(g, g)   # A, B: 21×21 격자의 α, β 값

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
# (왼쪽) 독립인 두 벡터: 방향이 서로 다르면 격자가 평면을 빈틈없이 채운다 → span = R^2.
v1, v2 = np.array([2., 1.]), np.array([-1., 2.])
P = A[..., None]*v1 + B[..., None]*v2   # 모든 (α,β)에 대한 αv1+βv2를 한 번에 계산
ax[0].scatter(P[..., 0], P[..., 1], s=6, color='#93C5FD')
arrow(ax[0], v1, '#2563EB', r'$\mathbf{v}_1$'); arrow(ax[0], v2, '#059669', r'$\mathbf{v}_2$')
ax[0].set_title('독립 → span = 평면 ($\\mathbb{R}^2$ 전체)')
# (오른쪽) 평행한 두 벡터(w2 = -2 w1): 계수를 아무리 바꿔도 결과가 한 직선을 못 벗어난다 → span = 직선.
w1, w2 = np.array([2., 1.]), np.array([-4., -2.])
Q = A[..., None]*w1 + B[..., None]*w2
ax[1].scatter(Q[..., 0], Q[..., 1], s=6, color='#FCA5A5')
arrow(ax[1], w1, '#2563EB', r'$\mathbf{w}_1$'); arrow(ax[1], w2, '#DC2626', r'$\mathbf{w}_2=-2\mathbf{w}_1$')
ax[1].set_title('평행 → span = 직선 (Vector 2개라도)')
for a_ in ax:
    a_.axhline(0, color='#ddd', lw=.8); a_.axvline(0, color='#ddd', lw=.8)
    a_.set_xlim(-7, 7); a_.set_ylim(-6, 6); a_.set_aspect('equal'); a_.grid(alpha=.3)
    a_.set_xlabel('$x_1$'); a_.set_ylabel('$x_2$')
plt.tight_layout(); plt.show()
print('→ 벡터가 2개여도 평행이면 span은 1차원(직선). 개수가 아니라 방향의 독립성이 차원을 정한다.')

## 3. Application: 신경망 한 층 = 가중 Linear combination

뉴런 한 층 $\mathbf{y}=W\mathbf{x}+\mathbf{b}$. 여기서 **$W\mathbf{x}$ = $W$ 열들의 Linear combination**(계수가 입력 $x_j$ — 3회차 Column picture). 같은 결과를 두 방식으로 계산해 일치를 확인한다.

In [ ]:
# 미니 선형층: 입력 3차원 → 출력 4차원. 신경망 한 층은 y = Wx + b 이다.
W = np.array([[ 1.0,  0.5, -1.0],
              [-2.0,  1.0,  0.5],
              [ 0.5,  2.0,  1.0],
              [ 1.0, -1.0,  0.5]])   # (4, 3): 행 = 출력뉴런 4개, 열 = 입력성분 3개
b = np.array([0.1, 0.0, -0.2, 0.3])  # 편향(bias)
x = np.array([2.0, -1.0, 3.0])       # 입력 벡터

# 같은 Wx+b를 두 방법으로 계산해 둘이 같음을 확인한다.
# 방법 A: 행렬·벡터 곱 그대로.
y_matrix  = W @ x + b
# 방법 B: W의 각 열에 입력 성분 x_j를 계수로 곱해 더한다(= Linear combination) + b.
y_lincomb = x[0]*W[:,0] + x[1]*W[:,1] + x[2]*W[:,2] + b
print('y (행렬·벡터 곱)            =', y_matrix)
print('y (열의 Linear combination) =', y_lincomb)
# 두 값이 같다는 것은 'Wx = W 열들의 x-계수 Linear combination'(3회차 Column picture)이라는 뜻이다.
assert np.allclose(y_matrix, y_lincomb)
print('✓ Wx + b = (W 열들의 x계수 Linear combination) + b  → 신경망 한 층의 본질')

In [ ]:
# [그림 3] 방법 B를 눈으로 본다: 출력뉴런마다 y가 '세 열의 기여 + 편향'으로 쌓인다.
# c_j = x_j * (W의 j번째 열). 세 기여를 더하면(+b) 빨간 막대 y와 정확히 같아진다.
c1, c2, c3 = x[0]*W[:,0], x[1]*W[:,1], x[2]*W[:,2]
idx = np.arange(4); width = 0.18
fig, ax = plt.subplots(figsize=(9, 5))
# 각 출력뉴런(0~3) 자리에 네 막대를 나란히: 세 열의 기여 c1,c2,c3와 최종 결과 y.
ax.bar(idx-1.5*width, c1, width, label=r'$x_1\mathbf{w}_{:,1}$', color='#2563EB')
ax.bar(idx-0.5*width, c2, width, label=r'$x_2\mathbf{w}_{:,2}$', color='#059669')
ax.bar(idx+0.5*width, c3, width, label=r'$x_3\mathbf{w}_{:,3}$', color='#D97706')
ax.bar(idx+1.5*width, y_matrix, width, label=r'$\mathbf{y}=W\mathbf{x}+\mathbf{b}$', color='#DC2626')
ax.axhline(0, color='#999', lw=.8)
ax.set_xticks(idx); ax.set_xticklabels([f'출력뉴런 {i}' for i in range(4)])
ax.set_title(r'$W\mathbf{x}+\mathbf{b}$ = $W$ 세 열의 가중 합 (+bias)')
ax.legend(fontsize=10); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

## 4. 정리

- 정의 1.1~1.5: Vector / 덧셈 / Scalar곱 / Linear combination / Span(직관)
- **덧셈 = 평행사변형, Scalar곱 = 같은 직선 위 신축·반전** (그림 1)
- **Span: 독립이면 평면, 평행이면 직선** — Vector 개수 ≠ Span 차원 (그림 2)
- **신경망 한 층 $W\mathbf{x}+\mathbf{b}$ = $W$ 열들의 가중 Linear combination + bias** (그림 3) — 정의 1.4가 그대로 AI 한 줄

**다음 회차(2회차)**: Norm·Dot product로 거리·각도·코사인 유사도. MNIST 이미지 비교가 본격 등장.